<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# TelcomCI — RH & Masse Salariale
## Notebook 2 — SQL Analytics : Masse Salariale, Turnover & Performance
### 📝 VERSION APPRENANT

> **Instructions :** Complète les cellules marquées `# TODO`.  
> Les blocs `MÉTHODE` t'expliquent l'approche attendue.  
> Les blocs `🧠 Tes observations` sont à remplir après exécution.

| | |
|---|---|
| **Prérequis** | Notebook 1 complété |
| **Niveau** | Avancé |
| **Outils** | Python, DuckDB (JupySQL), pandas |
| **Durée estimée** | 4h à 5h |

> **5 questions business** auxquelles ce notebook répond :
> 1. Quelle est l'**évolution de la masse salariale** mois par mois par département ?
> 2. Quels départements ont le **turnover le plus élevé** et quels en sont les motifs ?
> 3. Quels employés sont **les plus performants et sous-payés** par rapport à leur poste ?
> 4. Quel est le **coût réel du recrutement** par canal ?
> 5. **L'absentéisme a-t-il augmenté** en 2023 vs 2022 ?

---
## 0. Mise en place de l'environnement

In [33]:
!pip install jupysql==0.11.1 duckdb-engine --quiet

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb
import os, sys, warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)

COLORS = {
    'primary':   '#534AB7',
    'secondary': '#1D9E75',
    'warning':   '#EF9F27',
    'danger':    '#E24B4A',
    'neutral':   '#888780',
}
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F9F9F8',
    'axes.grid':        True,
    'grid.alpha':       0.35,
    'font.size':        11,
})

# ── Détection Colab / Local ──────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = '/content/drive/MyDrive/DataProjectLab/projects/rh_analytics/'
else:
    SAVE_PATH = './outputs/'
os.makedirs(SAVE_PATH, exist_ok=True)
print(f'📁 Environnement : {"Colab" if IN_COLAB else "Local"}')
print(f'📁 Dossier       : {SAVE_PATH}')
print('Configuration chargée ✅')

📁 Environnement : Local
📁 Dossier       : ./outputs/
Configuration chargée ✅


In [2]:
BASE_URL = 'https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/rh_analytics/data/'

# On recharge les tables depuis zéro — état propre et reproductible
conn = duckdb.connect()
conn.execute(f"""
    CREATE TABLE departements  AS SELECT * FROM read_csv_auto('{BASE_URL}departements.csv');
    CREATE TABLE employes      AS SELECT * FROM read_csv_auto('{BASE_URL}employes.csv',     timestampformat='%Y-%m-%d');
    CREATE TABLE salaires      AS SELECT * FROM read_csv_auto('{BASE_URL}salaires.csv');
    CREATE TABLE absences      AS SELECT * FROM read_csv_auto('{BASE_URL}absences.csv',     timestampformat='%Y-%m-%d');
    CREATE TABLE recrutements  AS SELECT * FROM read_csv_auto('{BASE_URL}recrutements.csv', timestampformat='%Y-%m-%d');
    CREATE TABLE evaluations   AS SELECT * FROM read_csv_auto('{BASE_URL}evaluations.csv');
""")
print('✅ 6 tables chargées')

%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
print('%%sql prêt ✅')

✅ 6 tables chargées
%%sql prêt ✅


---
## Étape 1 — Vues de nettoyage

### MÉTHODE — Principe de conservation des données brutes

On ne modifie jamais les tables sources. On crée des **vues** qui filtrent les anomalies détectées en NB1. Si on découvre une erreur dans la logique de filtrage, on corrige la vue sans toucher aux données originales.

> **DuckDB :** `CREATE OR REPLACE VIEW` remplace `CREATE OR ALTER VIEW` de SQL Server — syntaxe identique, même comportement.
>
> **Power BI se connectera aux vues**, pas aux tables brutes. Cela garantit que le dashboard reflète toujours des données propres.

In [36]:
%%sql
-- VUE 1 : employés propres
-- TODO : créer vw_employes_propres
--   → exclure salaire_base <= 0
--   → garder uniquement MIN(employe_id) par email (doublons)
CREATE OR REPLACE VIEW vw_employes_propres AS
SELECT * FROM employes
WHERE -- TODO

In [37]:
%%sql
-- VUE 2 : salaires valides
-- TODO : créer vw_salaires_valides
--   → exclure salaire_net <= 0
--   → garder uniquement statut_paiement = 'Payé'
CREATE OR REPLACE VIEW vw_salaires_valides AS
SELECT * FROM salaires
WHERE -- TODO

In [38]:
%%sql
-- VUE 3 : absences valides
-- TODO : créer vw_absences_valides → exclure nb_jours <= 0
CREATE OR REPLACE VIEW vw_absences_valides AS
SELECT * FROM absences
WHERE -- TODO

In [39]:
%%sql 
SELECT 'employes brut'         AS source, COUNT(*) AS nb FROM employes           UNION ALL
SELECT 'vw_employes_propres',   COUNT(*) FROM vw_employes_propres                 UNION ALL
SELECT 'salaires brut',         COUNT(*) FROM salaires                            UNION ALL
SELECT 'vw_salaires_valides',   COUNT(*) FROM vw_salaires_valides                 UNION ALL
SELECT 'absences brut',         COUNT(*) FROM absences                            UNION ALL
SELECT 'vw_absences_valides',   COUNT(*) FROM vw_absences_valides

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 2 — Masse salariale mensuelle avec LAG()

### MÉTHODE — LAG() : la fenêtre temporelle

`LAG(colonne, 1) OVER (PARTITION BY dept ORDER BY periode)` retourne la valeur de la ligne précédente dans la partition. En RH, c'est l'outil fondamental pour calculer 'combien la masse salariale a-t-elle changé ce mois vs le mois dernier ?'

**Pourquoi une CTE ?** La CTE `ms_mensuelle` pré-calcule la masse salariale par département/mois. La requête principale applique ensuite `LAG()` sur ce résultat. Sans CTE, on serait obligé d'imbriquer une sous-requête dans une autre — illisible.

In [40]:
%%sql  
WITH ms_mensuelle AS (
    SELECT
        s.departement_id,
        d.nom                                    AS departement,
        s.annee,
        s.mois,
        s.periode,
        SUM(s.salaire_brut + s.prime)            AS masse_salariale,
        COUNT(DISTINCT s.employe_id)             AS nb_employes
    FROM vw_salaires_valides s
    JOIN departements d ON s.departement_id = d.departement_id
    GROUP BY s.departement_id, d.nom, s.annee, s.mois, s.periode
)
select * from ms_mensuelle

In [41]:
%%sql  
-- Masse salariale mensuelle avec LAG() par département
-- TODO : construire la CTE ms_mensuelle
--   → SUM(salaire_brut + prime) AS masse_salariale
--   → GROUP BY departement_id, nom, annee, mois, periode
-- TODO : dans le SELECT principal
--   → LAG(masse_salariale) OVER (PARTITION BY departement_id ORDER BY periode)
--   → variation_pct = (masse - lag_masse) * 100.0 / NULLIF(lag_masse, 0)
--   → RANK() OVER (PARTITION BY annee, mois ORDER BY masse_salariale DESC)
WITH ms_mensuelle AS (
    SELECT
        s.departement_id,
        d.nom AS departement,
        s.annee, s.mois, s.periode
        -- TODO
    FROM vw_salaires_valides s
    JOIN departements d ON s.departement_id = d.departement_id
    GROUP BY s.departement_id, d.nom, s.annee, s.mois, s.periode
)
SELECT
    departement_id,
    departement, annee, mois, periode,
    masse_salariale, nb_employes
    -- TODO : LAG, variation_pct, rang_mensuel
FROM ms_mensuelle
ORDER BY departement, periode

In [42]:
%%sql 
-- Synthèse masse salariale par année et département
-- TODO : CTE ms_annuelle → SUM brut, primes, total par (département, annee)
-- TODO : SELECT → pct_annee avec SUM() OVER (PARTITION BY annee)
--                 → RANK() OVER (PARTITION BY annee ORDER BY masse_totale DESC)
WITH ms_annuelle AS (
    SELECT
        d.nom AS departement,
        s.annee
        -- TODO
    FROM vw_salaires_valides s
    JOIN departements d ON s.departement_id = d.departement_id
    GROUP BY d.departement_id, d.nom, s.annee
)
SELECT
    departement, annee, masse_totale
    -- TODO : pct_annee, rang
FROM ms_annuelle
ORDER BY annee, rang

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 3 — Taux de turnover avec CTEs et RANK()

### MÉTHODE — Calcul du taux de turnover

**Formule :** `Taux = Nb départs sur l'année / Effectif total du département × 100`

On utilise le dénominateur 'effectif total' (actifs + inactifs) qui inclut les départs eux-mêmes — méthode la plus conservative. **Architecture de la requête — 2 CTEs :**
- `departs_annee` : compte les départs par département, année et motif
- `effectif_total` : calcule l'effectif de référence par département

> **DuckDB :** `YEAR(date_depart)` fonctionne identiquement à SQL Server.

In [55]:
%%sql 
-- Taux de turnover par département et par année (2 CTEs)
-- CTE 1 : departs_annee → COUNT(*) par departement_id + YEAR(date_depart)
--         filtrer YEAR(date_depart) IN (2021, 2022, 2023)
-- CTE 2 : effectif_total → COUNT(*) par departement_id (table employes brute)
-- SELECT : taux = nb_departs * 100.0 / NULLIF(effectif, 0)
--          RANK() OVER (PARTITION BY annee ORDER BY taux DESC)
WITH departs_annee AS (
    -- TODO
),
effectif_total AS (
    -- TODO
)
SELECT
    d.nom AS departement
    -- TODO
FROM departs_annee da
JOIN effectif_total et ON da.departement_id = et.departement_id
JOIN departements d    ON da.departement_id = d.departement_id
ORDER BY da.annee, rang_risque

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

In [56]:
%%sql 
-- Motifs de départ par département
-- TODO : compter les départs par (département, motif_depart)
-- TODO : calculer pct_du_dept avec SUM() OVER (PARTITION BY d.nom)
SELECT
    d.nom AS departement,
    e.motif_depart
    -- TODO
FROM vw_employes_propres e
JOIN departements d ON e.departement_id = d.departement_id
WHERE e.date_depart IS NOT NULL
GROUP BY d.nom, e.motif_depart
ORDER BY d.nom, nb DESC

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

In [57]:
%%sql 
-- Coût estimé du turnover (2 CTEs + RANK)
-- Hypothèse : 6 mois de salaire net moyen par départ
-- CTE 1 : sal_net_moy → AVG(salaire_net) par departement_id
-- CTE 2 : departs_totaux → COUNT(*) par departement_id (WHERE date_depart IS NOT NULL)
-- SELECT : cout_turnover_estime = total_departs * salaire_net_moyen * 6
--          RANK() OVER (ORDER BY cout DESC)
WITH sal_net_moy AS (
    -- TODO
),
departs_totaux AS (
    -- TODO
)
SELECT
    d.nom AS departement
    -- TODO
FROM departs_totaux dt
JOIN sal_net_moy snm ON dt.departement_id = snm.departement_id
JOIN departements d  ON dt.departement_id = d.departement_id
ORDER BY rang_cout

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 4 — Matrice performance × salaire

### MÉTHODE — NTILE(4) et PERCENT_RANK()

**`NTILE(4)`** divise les employés en 4 groupes de taille égale selon leur salaire.
- Quartile 1 = 25% des salaires les plus bas
- Quartile 4 = 25% des salaires les plus élevés

**`PERCENT_RANK()`** calcule le rang relatif entre 0 et 1 :
```sql
PERCENT_RANK() OVER (ORDER BY note_globale DESC)
→ 0.0 = meilleur performeur | 0.1 = top 10% | 1.0 = moins bon
```

> **DuckDB :** `PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY col)` est supporté nativement — identique à SQL Server.

In [58]:
%%sql 
-- Matrice performance × salaire — détail par employé
-- CTE eval_2023 : note_globale + recommandation pour annee = 2023
-- CTE salaire_actuel : AVG(salaire_net) pour periode = '2024-06'
-- TODO : NTILE(4) OVER (ORDER BY salaire_net_actuel ASC) AS quartile_salaire
-- TODO : CASE PERCENT_RANK() OVER (ORDER BY note_globale DESC)
--           <= 0.10 → 'Top 10%' | <= 0.25 → 'Top 25%' | ELSE 'Standard'
-- Indice DuckDB : prenom || ' ' || nom
WITH eval_2023 AS (
    SELECT employe_id, note_globale, recommandation
    FROM evaluations WHERE annee = 2023
),
salaire_actuel AS (
    SELECT employe_id, ROUND(AVG(salaire_net), 0) AS salaire_net_actuel
    FROM vw_salaires_valides WHERE periode = '2024-06'
    GROUP BY employe_id
)
SELECT
    e.employe_id,
    e.prenom || ' ' || e.nom AS employe,
    d.nom AS departement,
    e.poste, e.type_contrat,
    sa.salaire_net_actuel,
    ev.note_globale, ev.recommandation
    -- TODO : quartile_salaire, segment_perf
FROM vw_employes_propres e
JOIN departements d             ON e.departement_id = d.departement_id
LEFT JOIN eval_2023 ev          ON e.employe_id     = ev.employe_id
LEFT JOIN salaire_actuel sa     ON e.employe_id     = sa.employe_id
WHERE e.statut = 'Actif'
ORDER BY ev.note_globale DESC, sa.salaire_net_actuel ASC

In [59]:
%%sql 
-- Synthèse par quadrant (PERCENTILE_CONT + CROSS JOIN)
-- CTE base : jointure employes × eval_2023 × salaire_actuel
-- CTE medianes : PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY col)
--   → med_note et med_sal
-- SELECT : CASE WHEN note >= med_note AND sal >= med_sal → 'Top Performer Bien Payé'
--               WHEN note >= med_note AND sal <  med_sal → 'Top Performer Sous-Payé'
--               WHEN note <  med_note AND sal >= med_sal → 'Faible Perf Bien Payé'
--               ELSE → 'À Accompagner'
-- TODO : compléter les CTEs et le SELECT final
WITH eval_2023 AS (
    SELECT employe_id, note_globale, recommandation
    FROM evaluations WHERE annee = 2023
),
salaire_actuel AS (
    SELECT employe_id, ROUND(AVG(salaire_net), 0) AS sal_net
    FROM vw_salaires_valides WHERE periode = '2024-06'
    GROUP BY employe_id
),
base AS (
    -- TODO : jointure + filtres statut = 'Actif', notes et salaires non NULL
),
medianes AS (
    -- TODO : PERCENTILE_CONT med_note et med_sal
)
SELECT
    -- TODO : quadrant, nb_employes, note_moy, sal_moy
FROM base b
CROSS JOIN medianes m
GROUP BY quadrant
ORDER BY nb_employes DESC

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

In [60]:
%%sql 
-- Alerte RH : Top performers sous-payés (Q1 salaire + note >= 4.0)
-- TODO : reprendre la structure de la requête précédente
-- TODO : ajouter NTILE(4) AS quartile_sal
-- TODO : filtrer WHERE quartile_sal = 1 AND note_globale >= 4.0
WITH eval_2023 AS (
    SELECT employe_id, note_globale, recommandation
    FROM evaluations WHERE annee = 2023
),
salaire_actuel AS (
    SELECT employe_id, ROUND(AVG(salaire_net), 0) AS sal_net
    FROM vw_salaires_valides WHERE periode = '2024-06'
    GROUP BY employe_id
),
quartiles AS (
    -- TODO
)
SELECT employe, departement, poste, sal_net, note_globale, recommandation, quartile_sal
FROM quartiles
WHERE -- TODO
ORDER BY note_globale DESC, sal_net ASC

---
## Étape 5 — Absentéisme

### MÉTHODE — pandas.pivot_table() remplace PIVOT SQL Server

`PIVOT FOR annee IN ([2021],[2022],[2023])` de SQL Server nécessite de lister les colonnes explicitement. `pandas.pivot_table()` est dynamique — s'adapte automatiquement et s'enchaîne directement avec seaborn pour la visualisation.

In [62]:
%%sql df_abs_raw <<
-- Données brutes pour le pivot
SELECT
    d.nom                              AS departement,
    YEAR(a.date_debut)                 AS annee,
    SUM(a.nb_jours)                    AS total_jours
FROM vw_absences_valides a
JOIN departements d ON a.departement_id = d.departement_id
WHERE YEAR(a.date_debut) IN (2021, 2022, 2023)
GROUP BY d.nom, YEAR(a.date_debut)
ORDER BY d.nom, annee

In [63]:
# Pivot pandas → heatmap absentéisme par département × année
# TODO : créer pivot_abs avec pandas.pivot_table()
#   index='departement', columns='annee', values='total_jours'
# TODO : ajouter colonne delta_2022_2023 = col[2023] - col[2022]
# TODO : afficher le pivot trié par delta décroissant
# TODO : visualiser avec sns.heatmap()
pivot_abs = df_abs_raw.pivot_table(
    # TODO
).fillna(0).astype(int)

pivot_abs['delta_2022_2023'] = # TODO

print(pivot_abs.sort_values('delta_2022_2023', ascending=False).to_string())

# TODO : visualisation heatmap avec sns.heatmap
fig, ax = plt.subplots(figsize=(10, 6))
# TODO
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}heatmap_absenteisme.png', dpi=150, bbox_inches='tight')
plt.show()

In [51]:
%%sql df_abs_injust <<
-- Taux d'absentéisme injustifié par département
-- TODO : nb_absences_injust (justifiee = false), jours_injustifies
-- TODO : total_jours_absences, pct_injustifiees
-- TODO : RANK() OVER (ORDER BY jours_injustifies DESC)
SELECT
    d.nom AS departement
    -- TODO
FROM vw_absences_valides a
JOIN departements d ON a.departement_id = d.departement_id
GROUP BY d.departement_id, d.nom
ORDER BY rang

In [64]:
df_abs_injust

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Étape 6 — Coût recrutement par canal

### MÉTHODE — Coût par embauche réelle

Le `cout_recrutement` est le coût du processus. Le vrai indicateur de performance est le **coût par embauche réelle** = coût total du canal / nombre d'embauches réussies. Un canal bon marché avec un faible taux de conversion peut coûter plus cher par embauche qu'un canal coûteux mais efficace.

In [65]:
%%sql 
-- Performance des canaux de recrutement
-- TODO : nb_recrutements, nb_embauches (SUM CAST embauche AS INTEGER)
-- TODO : taux_embauche_pct, duree_moy_jours, cout_moyen_process
-- TODO : cout_par_embauche = SUM(cout) / NULLIF(SUM(embauche), 0)
-- TODO : RANK() OVER (ORDER BY cout_par_embauche ASC) AS rang_efficacite
SELECT
    canal
    -- TODO
FROM recrutements
GROUP BY canal
ORDER BY rang_efficacite

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Rapport département — fonction Python (remplace la procédure stockée)

### MÉTHODE — Pourquoi une fonction Python ?

| Avantage | Détail |
|---|---|
| **Réutilisabilité** | `rapport_departement('DEP02')` — 1 ligne au lieu de 40 |
| **Flexibilité** | Peut enchaîner avec pandas et matplotlib |
| **Portabilité** | Fonctionne dans Colab et en local sans SQL Server |

> **DuckDB :** `DATE_DIFF('year', date_embauche, CURRENT_DATE)` remplace `DATEDIFF(YEAR, date_embauche, GETDATE())`.

In [66]:
def rapport_departement(dept_id: str, annee: int = 2023):
    """Rapport RH complet pour un département et une année.
    Remplace sp_rapport_rh_departement de SQL Server.
    """
    # TODO : requête 1 — KPIs annuels (effectif_actif, departs_annee, taux_turnover, anciennete_moy)
    # Indice : DATE_DIFF('year', date_embauche, CURRENT_DATE) pour l'ancienneté
    df_kpis = conn.execute(f"""
        -- TODO
    """).df()

    # TODO : requête 2 — Masse salariale de l'année (masse_salariale_annee, sal_brut_moyen, sal_net_moyen)
    df_sal = conn.execute(f"""
        -- TODO
    """).df()

    # TODO : requête 3 — Absentéisme (total_jours, jours_injustifies, pct_injustifiees)
    df_abs = conn.execute(f"""
        -- TODO
    """).df()

    # TODO : requête 4 — Note de performance (note_perf_moy, nb_evalue)
    df_perf = conn.execute(f"""
        -- TODO
    """).df()

    nom_dept = df_kpis.iloc[0]['departement'] if len(df_kpis) else dept_id
    print(f"{'='*50}")
    print(f"  RAPPORT RH : {nom_dept} — {annee}")
    print(f"{'='*50}")
    display(df_kpis)
    display(df_sal)
    display(df_abs)
    display(df_perf)

# Test : Technique & Réseau (DEP02) — 2023
rapport_departement('DEP02', 2023)

### 🧠 Tes observations

*(Rédige ici tes conclusions après avoir exécuté la cellule)*

- 
- 
- 

---
## Export des datasets analytiques pour Power BI

In [67]:
# Export des datasets produits pour Power BI
df_ms_annuelle.to_csv(f'{SAVE_PATH}telcomci_masse_salariale.csv', index=False)
df_turnover.to_csv(f'{SAVE_PATH}telcomci_turnover.csv', index=False)
df_perf_detail.to_csv(f'{SAVE_PATH}telcomci_performance.csv', index=False)
df_alerte_rh.to_csv(f'{SAVE_PATH}telcomci_alerte_rh.csv', index=False)
df_recru.to_csv(f'{SAVE_PATH}telcomci_recrutement.csv', index=False)

print('Fichiers exportés ✅')
print('  telcomci_masse_salariale.csv  → évolution masse salariale')
print('  telcomci_turnover.csv         → taux de turnover')
print('  telcomci_performance.csv      → matrice performance × salaire')
print('  telcomci_alerte_rh.csv        → employés sous-payés performants')
print('  telcomci_recrutement.csv      → coût par canal')
print(f'\n📁 Localisation : {SAVE_PATH}')

---
## Bilan du Notebook 2

### Réponses aux 5 questions DRH

| Question | Réponse |
|---|---|
| Évolution masse salariale | +16.8% (2021→2022) puis +15.0% (2022→2023) |
| Dept turnover le plus élevé | **Finance 8.6%** et **Marketing 6.7%** en 2023 |
| Employés sous-payés performants | **35 employés** (note moy 4.21, sal 622k FCFA) |
| Canal recrutement le plus efficace | **Cabinet** (taux 78.3%, coût/emb le plus bas) |
| Absentéisme en hausse 2023 | **Oui — Service Client +39j, Commercial +52j** |

### Patterns SQL maîtrisés

| Pattern | Utilisation |
|---|---|
| `CREATE OR REPLACE VIEW` | Vues de nettoyage non destructives |
| `LAG() OVER (PARTITION BY)` | Variation mensuelle masse salariale |
| `RANK() OVER (PARTITION BY)` | Classement dans un groupe |
| `NTILE(4) OVER` | Segmentation en quartiles salariaux |
| `PERCENT_RANK() OVER` | Rang relatif performance |
| `PERCENTILE_CONT(0.5)` | Calcul de la médiane |
| `pandas.pivot_table()` | Heatmap absentéisme (remplace PIVOT SQL Server) |
| Fonction Python | Rapport RH paramétré (remplace CREATE PROCEDURE) |
| `NULLIF(expr, 0)` | Protection division par zéro |
| `DATE_DIFF('year', d, CURRENT_DATE)` | Ancienneté (remplace DATEDIFF SQL Server) |

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.